# Electroweak (EWK) Corrections for ttbar Samples

The following implementation applies differential electroweak corrections to
ttbar events using the Rochester/HATHOR lookup tables stored in:

```python
HATHORMGPY_costheta.root
```

The correction is applied event-by-event as a function of:

- generator-level $m_{t\bar{t}}$
- generator-level top scattering angle in the $t\bar{t}$ rest frame

The nominal correction factor is defined as:

$$
\kappa(m_{t\bar{t}}, \cos\theta)
=
\frac{\mathrm{EWYt\_10}}
{\mathrm{EWno\_0}}
$$

where:

- `EWno_0` = LO QCD prediction
- `EWYt_10` = QCD + electroweak corrected prediction

The correction should be applied to the nominal ttbar event weights
before histogram filling.

---

# Nominal EWK Correction

```python
import numpy as np
import dask_awkward as dak
from coffea.lookup_tools import extractor


def electroweak_corrections(events,
                            rochester_file,
                            weight_factor):

    ext = extractor()

    ext.add_weight_sets([
        "lo_QCD EWno_0 " + rochester_file
    ])

    ext.add_weight_sets([
        "nlo_EWK EWYt_10 " + rochester_file
    ])

    ext.finalize()

    evaluator = ext.make_evaluator()

    mttbar = events["gen_ttbar_mass"]

    scattering_angle = events[
        "gen_top_scatteringangle_ttbarframe"
    ]

    lo_qcd = evaluator["lo_QCD"](
        mttbar,
        scattering_angle
    )

    nlo_ewk = evaluator["nlo_EWK"](
        mttbar,
        scattering_angle
    )

    kappa = nlo_ewk / lo_qcd

    kappa = dak.nan_to_num(kappa)

    return weight_factor * kappa
```

---

# Electroweak Uncertainty

The uncertainty is evaluated from the difference between multiplicative
and additive treatments of QCD and electroweak corrections:

$$
\delta_{\mathrm{EWK}}
=
1 + \delta_{\mathrm{QCD}}
\cdot
\delta_{\mathrm{EW}}
$$

with:

$$
\delta_{\mathrm{QCD}}
=
1 -
\frac{\sigma_{\mathrm{LO}}}
{\sigma_{\mathrm{POWHEG}}}
$$

and:

$$
\delta_{\mathrm{EW}}
=
\kappa - 1
$$

This produces a differential shape uncertainty in:

- $m_{t\bar{t}}$
- scattering angle

The uncertainty should be propagated as a shape nuisance.

---

# EWK Uncertainty Implementation

```python
import numpy as np
import dask_awkward as dak
from coffea.lookup_tools import extractor


def electroweak_uncertainty(events,
                            rochester_file,
                            weight_factor,
                            powheg_prediction):

    ext = extractor()

    ext.add_weight_sets([
        "lo_QCD EWno_0 " + rochester_file
    ])

    ext.add_weight_sets([
        "nlo_EWK EWYt_10 " + rochester_file
    ])

    ext.finalize()

    evaluator = ext.make_evaluator()

    mttbar = events["gen_ttbar_mass"]

    scattering_angle = events[
        "gen_top_scatteringangle_ttbarframe"
    ]

    lo_qcd = evaluator["lo_QCD"](
        mttbar,
        scattering_angle
    )

    nlo_ewk = evaluator["nlo_EWK"](
        mttbar,
        scattering_angle
    )

    kappa = nlo_ewk / lo_qcd

    kappa = dak.nan_to_num(kappa)

    delta_ewk = kappa - 1.0

    delta_qcd = (
        1.0 -
        (lo_qcd / (powheg_prediction + 1.0))
    )

    delta_qcd = dak.nan_to_num(delta_qcd)

    ewk_unc = 1.0 + (delta_qcd * delta_ewk)

    return weight_factor * ewk_unc
```

---

# Recommended Usage

## Nominal ttbar templates

```python
weight_nominal = electroweak_corrections(
    events,
    rochester_file,
    weight_nominal
)
```

## EWK uncertainty variations

```python
weight_EWKUp = weight_nominal * ewk_unc

weight_EWKDown = (
    weight_nominal / ewk_unc
)
```

or equivalently using a symmetrized prescription:

```python
weight_EWKDown = (
    weight_nominal * (2.0 - ewk_unc)
)
```

---

# Example Usage in Analysis

```python
if process == "TTbar":

    weight_nominal = electroweak_corrections(
        events,
        self.rochester_file,
        weight_nominal
    )

    ewk_unc = electroweak_uncertainty(
        events,
        self.rochester_file,
        np.ones_like(weight_nominal),
        powheg_prediction
    )

    weight_EWKUp = (
        weight_nominal * ewk_unc
    )

    weight_EWKDown = (
        weight_nominal / ewk_unc
    )
```

---

# Important Notes

- The correction must be applied using generator-level variables:
  - `gen_ttbar_mass`
  - `gen_top_scatteringangle_ttbarframe`

- The correction is independent of decay channel and can therefore be
  used consistently for:
  - dileptonic
  - semileptonic
  - all-hadronic ttbar analyses

- The same nuisance parameter should be correlated between semileptonic
  and dileptonic channels in combined fits.

- The correction should only be applied to the ttbar signal process.

- The resulting EWK uncertainty should be treated as a shape systematic
  uncertainty.

# NNLO QCD Reweighting for ttbar Samples

The following implementation applies NNLO QCD top-$p_T$ reweighting to
ttbar events using precomputed scale factor parameterizations.

The correction is applied event-by-event as a function of:

- generator-level top quark transverse momentum
- generator-level anti-top quark transverse momentum

The nominal NNLO QCD correction is evaluated using:

```python
NNPDF31_nnlo_as_0118
```

together with one of the available scale factor parameterizations:

- `SF_1`
- `SF_2`
- `SF_3`

In the current implementation, the nominal recommendation is:

```python
SF_3
```

The correction is applied multiplicatively to the nominal event weights
before histogram filling. :contentReference[oaicite:0]{index=0}

---

# NNLO QCD Reweighting

The event weight is modified according to:

$$
w_{\mathrm{NNLO}}
=
w_{0}
\times
\sqrt{
SF(p_T^t)
\cdot
SF(p_T^{\bar{t}})
}
$$

where:

- $w_0$ = original event weight
- $SF(p_T)$ = NNLO QCD scale factor function

The geometric mean of the top and anti-top scale factors is used.

---

# NNLO QCD Reweighting Implementation

```python
import numpy as np
import awkward as ak
import dask_awkward as dak


def nnlo_qcd_reweighting(pdf,
                         sf_function_name,
                         events,
                         weight_factor):

    params = np.load(
        f"{main_config['fw_dir']}"
        f"/data/top_pt_reweighting/"
        f"nnlo_qcd_{pdf}_{sf_function_name}.npy",
        allow_pickle=True
    )

    params = np.array([
        params[i].nominal_value
        for i in range(len(params))
    ])

    if sf_function_name == "SF_1":
        sf_function = sf1

    elif sf_function_name == "SF_2":
        sf_function = sf2

    elif sf_function_name == "SF_3":
        sf_function = sf3

    else:
        raise NotImplementedError(
            f"Scale factor function "
            f"{sf_function_name} "
            f"is not implemented."
        )

    top_sfs = sf_function(
        events["gen_top_pt"],
        *params
    )

    tbar_sfs = sf_function(
        events["gen_tbar_pt"],
        *params
    )

    min_sf = sf_function(500, *params)

    top_sfs = ak.where(
        top_sfs < min_sf,
        min_sf,
        top_sfs
    )

    tbar_sfs = ak.where(
        tbar_sfs < min_sf,
        min_sf,
        tbar_sfs
    )

    toppt_scalefactor = np.sqrt(
        top_sfs * tbar_sfs
    )

    return toppt_scalefactor * weight_factor
```

---

# Scale Factor Parameterizations

## SF\_1

The first parameterization is defined as:

$$
SF_1(x)
=
e^{p_0 x + p_1}
$$

```python
def sf1(x, p0, p1):

    return np.e ** (p0 * x + p1)
```

---

## SF\_2

The second parameterization is defined as:

$$
SF_2(x)
=
p_0 e^{p_1 x}
+
p_2 x
+
p_3
$$

```python
def sf2(x, p0, p1, p2, p3):

    return (
        p0 * (np.e ** (p1 * x))
        + p2 * x
        + p3
    )
```

---

## SF\_3

The nominal parameterization currently used is:

$$
SF_3(x)
=
p_0 e^{p_1 x}
+
p_2
$$

```python
def sf3(x, p0, p1, p2):

    return (
        p0 * (np.e ** (p1 * x))
        + p2
    )
```

---

# Recommended Usage

## Nominal ttbar templates

```python
weight_nominal = nnlo_qcd_reweighting(
    "NNPDF31_nnlo_as_0118",
    "SF_3",
    events,
    weight_nominal
)
```

---

# Example Usage in Analysis

```python
if process == "TTbar":

    weight_nominal = nnlo_qcd_reweighting(
        "NNPDF31_nnlo_as_0118",
        "SF_3",
        events,
        weight_nominal
    )
```

---

# Important Notes

- The correction must be applied using generator-level variables:
  - `gen_top_pt`
  - `gen_tbar_pt`

- The correction is independent of decay channel and can therefore be
  used consistently for:
  - dileptonic
  - semileptonic
  - all-hadronic ttbar analyses

- The correction should only be applied to the ttbar signal process.

- The resulting correction modifies both normalization and shape,
  particularly in boosted top kinematic regions.

- The same nuisance parameter should be correlated between semileptonic
  and dileptonic channels in combined fits.